In [ ]:
import math

def solve_two_stage(U, V2, V1, lambda_, mp_ratio=None, mp_fixed=None):
    # Вычисляем отношения скоростей
    r1 = math.exp(V1 / U)          # exp(V1/U)
    r2 = math.exp((V2 - V1) / U)   # exp((V2-V1)/U)

    # Если задана фиксированная полезная нагрузка, mp известна
    if mp_fixed is not None:
        mp = mp_fixed
        # Масса второй ступени
        M2 = mp * (1 - r2) / (lambda_ * r2 - 1)
        # Масса первой ступени
        M2_plus_mp = M2 + mp
        M1 = M2_plus_mp * (1 - r1) / (lambda_ * r1 - 1)
        m0 = M1 + M2 + mp
        return {
            'M1': M1,
            'M2': M2,
            'mp': mp,
            'm0': m0,
            'M1_ratio': M1 / m0,
            'M2_ratio': M2 / m0,
            'mp_ratio': mp / m0
        }
    # Если задано отношение mp/m0
    elif mp_ratio is not None:
        # Решаем относительно долей от m0
        # Обозначим x = M1/m0, y = M2/m0, p = mp/m0
        p = mp_ratio
        # Из уравнения для второй ступени: y = p * (1 - r2) / (λ r2 - 1)
        y = p * (1 - r2) / (lambda_ * r2 - 1)
        # Из уравнения для первой ступени: x = (1/r1 - y - p) / λ
        x = (1.0 / r1 - y - p) / lambda_
        # Проверка суммы
        total = x + y + p
        if abs(total - 1.0) > 1e-9:
            print(f"Предупреждение: сумма долей x+y+p = {total:.6f} (должна быть 1)")
        return {
            'M1_ratio': x,
            'M2_ratio': y,
            'mp_ratio': p,
            'm0_ratio': 1.0
        }
    else:
        raise ValueError("Необходимо задать либо mp_fixed, либо mp_ratio")

def solve_three_stage(U, V3, V2, V1, lambda_, mp_ratio=None, mp_fixed=None):
    r1 = math.exp(V1 / U)
    r2 = math.exp((V2 - V1) / U)
    r3 = math.exp((V3 - V2) / U)

    if mp_fixed is not None:
        mp = mp_fixed
        # Третья ступень
        M3 = mp * (1 - r3) / (lambda_ * r3 - 1)
        # Вторая ступень
        M3_plus_mp = M3 + mp
        M2 = M3_plus_mp * (1 - r2) / (lambda_ * r2 - 1)
        # Первая ступень
        M2_plus_M3_plus_mp = M2 + M3 + mp
        M1 = M2_plus_M3_plus_mp * (1 - r1) / (lambda_ * r1 - 1)
        m0 = M1 + M2 + M3 + mp
        return {
            'M1': M1, 'M2': M2, 'M3': M3, 'mp': mp, 'm0': m0,
            'M1_ratio': M1 / m0, 'M2_ratio': M2 / m0, 'M3_ratio': M3 / m0,
            'mp_ratio': mp / m0
        }
    elif mp_ratio is not None:
        p = mp_ratio
        # Выражаем доли аналогично двухступенчатому случаю
        y3 = p * (1 - r3) / (lambda_ * r3 - 1)   # M3/m0
        y2 = ( (y3 + p) * (1 - r2) ) / (lambda_ * r2 - 1)  # M2/m0
        y1 = ( (y2 + y3 + p) * (1 - r1) ) / (lambda_ * r1 - 1)  # M1/m0
        total = y1 + y2 + y3 + p
        if abs(total - 1.0) > 1e-9:
            print(f"Предупреждение: сумма долей = {total:.6f} (должна быть 1)")
        return {
            'M1_ratio': y1, 'M2_ratio': y2, 'M3_ratio': y3,
            'mp_ratio': p, 'm0_ratio': 1.0
        }
    else:
        raise ValueError("Задайте mp_fixed или mp_ratio")

def main():
    print("=" * 60)
    print("Расчёт масс ступеней ракеты")
    print("=" * 60)

    # Параметры из задания
    U = 3.0          # км/с
    lambda_ = 0.1
    V2_target = 8.0  # км/с (конечная скорость для двухступенчатой)
    # Для двухступенчатой: V1 = V2/2
    V1_two = V2_target / 2.0   # 4 км/с
    mp_ratio_given = 1.0 / 20.0   # mp = m0/20

    print("\n--- Двухступенчатая ракета ---")
    print(f"U = {U} км/с, λ = {lambda_}, V2 = {V2_target} км/с, V1 = V2/2 = {V1_two} км/с")
    print(f"mp/m0 = {mp_ratio_given} (mр = m0/20)")

    # Расчёт в долях от m0
    result_two = solve_two_stage(U, V2_target, V1_two, lambda_, mp_ratio=mp_ratio_given)
    print("\nРезультат (в долях от стартовой массы m0):")
    print(f"  M1/m0 = {result_two['M1_ratio']:.6f}")
    print(f"  M2/m0 = {result_two['M2_ratio']:.6f}")
    print(f"  mp/m0 = {result_two['mp_ratio']:.6f}")

    # Если доли положительны и сумма = 1, можно выразить через mp
    if result_two['M1_ratio'] > 0 and result_two['M2_ratio'] > 0:
        print("\nВыражение через полезную нагрузку (mp):")
        M1_over_mp = result_two['M1_ratio'] / result_two['mp_ratio']
        M2_over_mp = result_two['M2_ratio'] / result_two['mp_ratio']
        print(f"  M1 = {M1_over_mp:.4f} * mp")
        print(f"  M2 = {M2_over_mp:.4f} * mp")
        print(f"  m0 = {1/result_two['mp_ratio']:.4f} * mp")
        # Проверка условия m0 = 20*mp
        print(f"  Условие mp = m0/20 даёт m0/mp = 20, получено m0/mp = {1/result_two['mp_ratio']:.4f}")
        if abs(1/result_two['mp_ratio'] - 20) > 0.01:
            print("  Внимание: заданное условие mp = m0/20 не выполняется для выбранных V1, V2, λ.")
            print("  Требуется изменить параметры (например, V1).")
    else:
        print("\nОтрицательные доли масс: при заданных параметрах физически реализуемого решения нет.")
        print("Попробуйте изменить V1 или λ.")

    # Трёхступенчатая ракета
    print("\n" + "=" * 60)
    print("--- Трёхступенчатая ракета ---")
    V3_target = 8.0   # км/с
    V2_three = V3_target / 2.0   # 4 км/с
    V1_three = V2_three / 2.0    # 2 км/с
    print(f"U = {U} км/с, λ = {lambda_}, V3 = {V3_target} км/с")
    print(f"V2 = V3/2 = {V2_three} км/с, V1 = V2/2 = {V1_three} км/с")
    print(f"mp/m0 = {mp_ratio_given} (mр = m0/20)")

    result_three = solve_three_stage(U, V3_target, V2_three, V1_three, lambda_, mp_ratio=mp_ratio_given)
    print("\nРезультат (в долях от m0):")
    print(f"  M1/m0 = {result_three['M1_ratio']:.6f}")
    print(f"  M2/m0 = {result_three['M2_ratio']:.6f}")
    print(f"  M3/m0 = {result_three['M3_ratio']:.6f}")
    print(f"  mp/m0 = {result_three['mp_ratio']:.6f}")

    if result_three['M1_ratio'] > 0 and result_three['M2_ratio'] > 0 and result_three['M3_ratio'] > 0:
        print("\nВыражение через mp:")
        M1_over_mp = result_three['M1_ratio'] / result_three['mp_ratio']
        M2_over_mp = result_three['M2_ratio'] / result_three['mp_ratio']
        M3_over_mp = result_three['M3_ratio'] / result_three['mp_ratio']
        print(f"  M1 = {M1_over_mp:.4f} * mp")
        print(f"  M2 = {M2_over_mp:.4f} * mp")
        print(f"  M3 = {M3_over_mp:.4f} * mp")
        print(f"  m0 = {1/result_three['mp_ratio']:.4f} * mp")
        if abs(1/result_three['mp_ratio'] - 20) > 0.01:
            print("  Условие mp = m0/20 не выполняется (требуется корректировка параметров).")
    else:
        print("\nНевозможно получить положительные массы при заданных параметрах.")

    # Дополнительно: вывод общих формул для трёхступенчатой ракеты
    print("\n" + "=" * 60)
    print("Формулы для трёхступенчатой ракеты (по индукции):")
    print("""
    Пусть:
      m0 = M1 + M2 + M3 + mp  — стартовая масса,
      mp — масса полезной нагрузки,
      M1, M2, M3 — массы ступеней,
      λ — доля отбрасываемой массы конструкции,
      U — скорость истечения газов.

    Скорости после каждой ступени:
      V1 = U * ln( (M1+M2+M3+mp) / (λ·M1 + M2+M3+mp) )
      V2 = V1 + U * ln( (M2+M3+mp) / (λ·M2 + M3+mp) )
      V3 = V2 + U * ln( (M3+mp) / (λ·M3 + mp) )

    Обратные соотношения для масс:
      r1 = exp(V1/U),   r2 = exp((V2-V1)/U),   r3 = exp((V3-V2)/U)
      M3 = mp * (1 - r3) / (λ·r3 - 1)
      M2 = (M3+mp) * (1 - r2) / (λ·r2 - 1)
      M1 = (M2+M3+mp) * (1 - r1) / (λ·r1 - 1)
    """)

if __name__ == "__main__":
    main()

Расчёт масс ступеней ракеты

--- Двухступенчатая ракета ---
U = 3.0 км/с, λ = 0.1, V2 = 8.0 км/с, V1 = V2/2 = 4.0 км/с
mp/m0 = 0.05 (mр = m0/20)
Предупреждение: сумма долей x+y+p = 0.160378 (должна быть 1)

Результат (в долях от стартовой массы m0):
  M1/m0 = -0.114688
  M2/m0 = 0.225066
  mp/m0 = 0.050000

Отрицательные доли масс: при заданных параметрах физически реализуемого решения нет.
Попробуйте изменить V1 или λ.

--- Трёхступенчатая ракета ---
U = 3.0 км/с, λ = 0.1, V3 = 8.0 км/с
V2 = V3/2 = 4.0 км/с, V1 = V2/2 = 2.0 км/с
mp/m0 = 0.05 (mр = m0/20)
Предупреждение: сумма долей = 1.303602 (должна быть 1)

Результат (в долях от m0):
  M1/m0 = 0.704789
  M2/m0 = 0.323747
  M3/m0 = 0.225066
  mp/m0 = 0.050000

Выражение через mp:
  M1 = 14.0958 * mp
  M2 = 6.4749 * mp
  M3 = 4.5013 * mp
  m0 = 20.0000 * mp

Формулы для трёхступенчатой ракеты (по индукции):

    Пусть:
      m0 = M1 + M2 + M3 + mp  — стартовая масса,
      mp — масса полезной нагрузки,
      M1, M2, M3 — массы ступене